# S1 · Every kind of hyperindex, one input set
Each version's address function is run on the same two inputs: **(A)** all 18,278 lowercase words of length ≤ 3 (collision counts are exact, not sampled) and **(B)** the first 1,000 characters of the corpus sample (size). Versions are loaded by path, unmodified.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys, string, re, math
from itertools import product
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop")); sys.path.insert(0, os.path.join(PROJ, "VAPMIP", "engines")); sys.path.insert(0, PROJ)
D = "PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/"
fa   = load("PtolemyDesktop/Archive/ainulindale_first_age/FA_smnnip_hyperindex.py", "fa")
hw2  = load(D + "hyperwebster.py", "hw2").HyperWebster()                    # v02
hh   = load(D + "hyperwebster-hash.py", "hh").HyperWebster()                # v03
from Callimachus.v09.core.hyperwebster import HyperWebster as HW9           # v09
hw9  = HW9()
sw   = load("SemanticWordEngine/hyperwebster.py", "sw"); mo = load("VAPMIP/monad.py", "mo")
lex  = sw.Lexicon({k: v for k, v in mo._ZERO_CACHE.items() if k <= 50}, path="/dev/null")   # v11 with 50 zeros
e17  = load("VAPMIP/engines/e17_hyperindexing.py", "e17"); prim16 = e17.primes_up_to_n(16)
vq   = load("ValaQuenta/modules/hyperwebster/maths.py", "vq")
mi   = load("PtolemyDesktop/Archimedes/Maths/MidlineIndex.py", "mi")
words = ["".join(p) for n in (1, 2, 3) for p in product(string.ascii_lowercase, repeat=n)]
print(len(words), "words; modules loaded")

18278 words; modules loaded


## The address function of each version, as one callable
Six of the versions are exact or nearly so; three are deliberately many-to-one. The **multiset** address (v06 notebook 03) is written out here so the row can be measured.

In [3]:
from collections import Counter
idx = {c: i for i, c in enumerate(hw2.characters)}
def multiset_rank(text):
    c = Counter(text); syms = sorted(c, key=lambda ch: idx[ch]); m = len(text)
    P = math.factorial(m)
    for k in c.values(): P //= math.factorial(k)
    r = 0
    for ch in text:
        r += P * sum(c[x] for x in syms if idx[x] < idx[ch]) // m
        P = P * c[ch] // m; c[ch] -= 1; m -= 1
    return r
def fsig(w):
    raw = [e17.project(w, k, prim16) for k in range(16)]; nm = math.sqrt(sum(x*x for x in raw)); v = [x/nm for x in raw]
    return tuple(sorted(range(16), key=lambda k: abs(v[k])))
KINDS = [  # (version, kind, function word -> hashable address)
 ("v00", "mixed radix, length NOT included",   lambda w: fa.TextHyperIndex.encode_text(w)),
 ("v00", "mixed radix + stored length",         lambda w: (fa.TextHyperIndex.encode_text(w), len(w))),
 ("v02", "exact bijection (shortlex rank)",     lambda w: hw2.point_to_text(w)),
 ("v03", "SHA-256 label of the bijection",      lambda w: hh.index_text(w).label),
 ("v09", "SHA-256 label, Unicode order",        lambda w: hw9.label_of(w)),
 ("v14", "mixed radix + stored length",         lambda w: (vq.horner_encode(w), len(w))),
 ("v06", "(count vector, rank in anagram class)", lambda w: (tuple(sorted(Counter(w).items())), multiset_rank(w))),
 ("v11", "semantic zero ordinal (50 zeros)",    lambda w: lex.address_of(w)[0]),
 ("v12", "prime zero index",                    lambda w: mo._word_zero_idx(w)),
 ("v13", "firing signature, n = 16",            fsig),
 ("v19", "Midline base-96 (case exact)",         lambda w: mi.encode_text(w)),
]
rows = []
for ver, kind, f in KINDS:
    t0 = time.perf_counter(); vals = [f(w) for w in words]; dt = time.perf_counter() - t0
    d = len(set(vals))
    rows.append((ver, kind, d, len(words) - d, f("listen") == f("silent"), dt))
print("%-4s %-42s %9s %10s %-18s %s" % ("ver", "kind", "distinct", "collisions", "listen==silent?", "secs"))
for ver, kind, d, c, an, dt in rows: print("%-4s %-42s %9d %10d %-18s %.1f" % (ver, kind, d, c, an, dt))

ver  kind                                        distinct collisions listen==silent?    secs
v00  mixed radix, length NOT included               17576        702 False              0.0
v00  mixed radix + stored length                    18278          0 False              0.0
v02  exact bijection (shortlex rank)                18278          0 False              0.0
v03  SHA-256 label of the bijection                 18278          0 False              0.1
v09  SHA-256 label, Unicode order                   18278          0 False              0.0
v14  mixed radix + stored length                    18278          0 False              0.0
v06  (count vector, rank in anagram class)          18278          0 False              0.3
v11  semantic zero ordinal (50 zeros)                  23      18255 True               0.1
v12  prime zero index                                2409      15869 False              0.1
v13  firing signature, n = 16                           8      18270 False     

**Reading the table.** *Distinct* counts addresses among 18,278 words (the exact rows reach 18,278). Rows with **collisions > 0** cannot be inverted. v00's integer alone collides because symbol 0 is `a`; with the length it is exact. v19 is exact on these words (case is kept, the 96-symbol domain covers lowercase).

In [4]:
assert dict((k, d) for _, k, d, *_ in rows)["exact bijection (shortlex rank)"] == len(words)
ex = {kind: (d == len(words)) for _, kind, d, *_ in rows}
print("exact on this set:", [k for k, v in ex.items() if v])
print("many-to-one on this set:", [k for k, v in ex.items() if not v])

exact on this set: ['mixed radix + stored length', 'exact bijection (shortlex rank)', 'SHA-256 label of the bijection', 'SHA-256 label, Unicode order', '(count vector, rank in anagram class)', 'Midline base-96 (case exact)']
many-to-one on this set: ['mixed radix, length NOT included', 'semantic zero ordinal (50 zeros)', 'prime zero index', 'firing signature, n = 16']


## Size, on 1,000 characters of the corpus sample
Bits of the address (where the address grows with the input) or bits of its range (where it is fixed width).

In [5]:
t = CORPUS[:1000]; n = len(t); N = 97
mult = math.factorial(n)
for k in Counter(t).values(): mult //= math.factorial(k)
dom = "".join(c for c in mi.normalise(t) if 0x20 <= ord(c) <= 0x7E or c == chr(10))
sizes = [
 ("raw, 8 bits per character",                     8 * n, "grows"),
 ("v02 exact bijection",                            hw2.point_to_text(t).bit_length(), "grows"),
 ("v00/v14 mixed radix + length",                   fa.TextHyperIndex.encode_text(t).bit_length() if False else vq.horner_encode(t).bit_length(), "grows"),
 ("v03/v09 label (SHA-256)",                        256, "fixed (key only; payload is the v02 address)"),
 ("v06 (count vector, rank)",                       (mult.bit_length() - 1) + (math.comb(n + N - 1, N - 1).bit_length() - 1), "grows (order-0 entropy)"),
 ("v11 zero ordinal (of 50)",                       round(math.log2(50), 2), "fixed"),
 ("v12 zero index (of 6,543)",                      round(math.log2(6543), 2), "fixed"),
 ("v13 signature, n = 16 (16! values)",             round(math.log2(math.factorial(16)), 2), "fixed"),
 ("v19 Midline, %d characters" % len(dom),          mi.encode_text(dom).bit_length(), "grows (domain 96)"),
 ("v16 fold, rungs ℂ…𝕊 (from v16 notebook)",        64, "fixed"),
]
print("%-46s %10s   %s" % ("address", "bits", "behaviour"))
for name, b, beh in sizes: print("%-46s %10s   %s" % (name, b, beh))

address                                              bits   behaviour
raw, 8 bits per character                            8000   grows
v02 exact bijection                                  6600   grows
v00/v14 mixed radix + length                         6600   grows
v03/v09 label (SHA-256)                               256   fixed (key only; payload is the v02 address)
v06 (count vector, rank)                             5126   grows (order-0 entropy)
v11 zero ordinal (of 50)                             5.64   fixed
v12 zero index (of 6,543)                           12.68   fixed
v13 signature, n = 16 (16! values)                  44.25   fixed
v19 Midline, 1000 characters                         6585   grows (domain 96)
v16 fold, rungs ℂ…𝕊 (from v16 notebook)                64   fixed


Two families follow from the table: **addresses that grow with the input** (exact; the size is the text's information content in another radix; the only ones smaller than raw are those that give up part of the alphabet or spend the text's non-uniformity) and **addresses of fixed width** (v03/v09 label, v11, v12, v13, v16), which are keys, not stores: they identify or locate, and none of them inverts. This is the paper's classification.